# EGC5310 — Semana 06 · Notebook Estudante

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/professorviniciusramos/EGC5310-EstruturasDados/blob/main/Semana06/EGC5310-EstruturasDados-S06-04-Estudante.ipynb)

## Representação, operações e custo

Neste notebook, **não estamos procurando uma estrutura vencedora**. Vamos manter os mesmos dados e mudar as perguntas para observar como a representação interfere no trabalho necessário.

Em cada atividade:

**preveja → implemente → execute → interprete → explique o mecanismo.**

Não execute benchmarks antes de registrar sua previsão. Tempos absolutos variam entre computadores; compare tendências e relações dentro da sua própria máquina.


## Preparação — o dataset real

Usaremos o **Online Retail da UCI**. Cada linha representa um item de uma transação, portanto `StockCode` aparece repetidamente.

Isso é importante: quando perguntarmos **“qual é a descrição deste produto?”**, criaremos uma visão deduplicada por código. Quando perguntarmos sobre clientes ou processamento em massa, voltaremos ao dataset transacional completo.

Execute as duas células de preparação antes da Atividade 1.


In [ ]:
import io, zipfile, urllib.request, pathlib
import pandas as pd
import numpy as np

URL = "https://archive.ics.uci.edu/static/public/352/online%2Bretail.zip"
ARQUIVO = pathlib.Path("Online Retail.xlsx")

if not ARQUIVO.exists():
    try:
        with urllib.request.urlopen(URL, timeout=90) as resposta:
            pacote = resposta.read()
        with zipfile.ZipFile(io.BytesIO(pacote)) as zipado:
            nome = next(nome for nome in zipado.namelist() if nome.lower().endswith(".xlsx"))
            ARQUIVO.write_bytes(zipado.read(nome))
    except Exception as erro:
        print("Download indisponível:", erro)
        print("Baixe o arquivo na página da UCI e envie Online Retail.xlsx para a sessão.")
        if 'google.colab' in __import__('sys').modules:
            from google.colab import files
            files.upload()
        if not ARQUIVO.exists():
            raise RuntimeError("Envie Online Retail.xlsx e execute novamente esta célula") from erro

# O Excel original tem aproximadamente 541 mil linhas.
df = pd.read_excel(ARQUIVO, engine="openpyxl")
print("Linhas:", len(df))
print("Colunas:", list(df.columns))
display(df.head(4))


In [ ]:
# Uma linha do dataset representa um item de uma fatura.
# O mesmo StockCode pode aparecer em muitas transações.
#
# Para os desafios de busca por PRODUTO, construiremos uma visão com
# uma entrada por código. Isto NÃO substitui o dataset transacional.

produtos_por_codigo = {}
for codigo, descricao in zip(df["StockCode"], df["Description"]):
    if pd.notna(codigo) and pd.notna(descricao):
        produtos_por_codigo[str(codigo)] = str(descricao)

registros = list(produtos_por_codigo.items())

print("Linhas transacionais:", len(df))
print("Produtos distintos na visão código → descrição:", len(registros))
print("Exemplo:", registros[:5])


## Atividade 1 — escolha antes da medida

A loja precisa executar cinco tipos de trabalho:

1. recuperar a descrição de um produto a partir de seu código;
2. recuperar todos os códigos dentro de uma faixa;
3. acrescentar novos produtos;
4. comparar grupos de clientes;
5. calcular valores sobre centenas de milhares de transações.

### Antes de programar

Para **cada operação**, registre:

- qual representação/estrutura você escolheria;
- qual propriedade dessa estrutura motivou a escolha;
- se existe algum custo de preparação;
- qual é sua principal dúvida sobre a escolha.

Não há obrigação de escolher uma única estrutura para tudo.

> **Minha hipótese inicial**
>
> Consulta exata →  
> Intervalo →  
> Inserção →  
> Grupos de clientes →  
> Processamento em massa →


In [ ]:
operacoes = [
    "código exato",
    "intervalo",
    "inserção",
    "grupos de clientes",
    "processamento em massa"
]

for operacao in operacoes:
    print(f"{operacao:24s} → minha escolha: __________________")


## Atividade 2 — os mesmos pares, três formas de buscar

Agora a pergunta é específica:

> **Dado um `StockCode`, qual é sua descrição?**

Vamos representar os mesmos pares `código → descrição` de três maneiras:

- lista não ordenada → busca sequencial;
- lista ordenada → busca binária;
- `dict` → hashing.

### Antes de executar

1. Qual estratégia pode precisar percorrer quase todos os produtos?
2. Qual exige uma propriedade adicional antes da consulta?
3. Qual exige construir um índice por hashing?
4. O que você espera que aconteça para uma chave **ausente**?

Implemente as partes marcadas com `TODO`. Depois teste uma chave presente e uma ausente.


In [ ]:
def busca_sequencial(registros, alvo):
    for codigo, descricao in registros:
        # TODO: se encontrou o código, devolva a descrição
        pass
    return None


def busca_binaria(registros_ordenados, alvo):
    inicio = 0
    fim = len(registros_ordenados) - 1

    while inicio <= fim:
        meio = (inicio + fim) // 2
        codigo, descricao = registros_ordenados[meio]

        # TODO 1: encontrou?
        # TODO 2: se codigo < alvo, qual limite muda?
        # TODO 3: caso contrário, qual limite muda?
        pass

    return None


ordenada = sorted(registros)
indice = dict(registros)

alvo_presente = registros[-1][0]
alvo_ausente = "CODIGO_AUSENTE"

# TODO: consulte os dois alvos com as três estratégias.
# Os resultados para cada alvo devem ser equivalentes.


### Interprete antes de seguir

Preencha depois de executar:

> Na busca sequencial, o trabalho cresce porque…  
> Na busca binária, a ordenação é necessária porque…  
> No `dict`, não percorremos a lista porque…  
> Para a chave ausente, observei que…  

**Não conclua ainda que `dict` é “melhor”.** Até aqui fizemos apenas uma pergunta: busca por chave exata.


## Atividade 3 — preparação + muitas consultas

Uma consulta isolada não conta toda a história.

Uma representação pode exigir trabalho antes da primeira consulta:

- lista: quase nenhum preparo adicional;
- lista ordenada: precisa ser ordenada;
- `dict`: precisa ser construído.

### Hipótese

Se fizermos `q` consultas, pense em:

**custo total = preparação + custo das q consultas**

Antes de medir, responda:

> Com 1 consulta, espero que…  
> Com 100 consultas, espero que…  
> Com 10.000 consultas, espero que…  
> Quando `n` cresce, espero que…


In [ ]:
# Complete e execute depois da previsão.
import timeit
import random
import statistics

def medir_buscas(registros,
                 tamanhos=(500, 2000, 4000),
                 quantidades=(1, 10, 100, 1000, 10000)):
    saida = []
    aleatorio = random.Random(5310)

    for n in tamanhos:
        base = registros[:min(n, len(registros))]
        chaves = [codigo for codigo, _ in base]
        if not chaves:
            continue

        consultas = [
            chaves[aleatorio.randrange(len(chaves))] if i % 2
            else "CODIGO_AUSENTE"
            for i in range(max(quantidades))
        ]

        # TODO: para cada estratégia, medir SEPARADAMENTE:
        # 1. preparação
        # 2. lote de consultas
        # 3. total = preparação + consultas
        #
        # Use timeit.repeat(..., repeat=3, number=1)
        # e a mediana com statistics.median(...).

    return pd.DataFrame(
        saida,
        columns=["n", "consultas", "estrutura",
                 "preparo_s", "consultas_s", "total_s"]
    )

# resultado_buscas = medir_buscas(registros)
# display(resultado_buscas)


### Interprete o benchmark

Não procure apenas “quem foi mais rápido”.

Responda:

1. O custo de preparação aparece de forma relevante em quais estratégias?
2. O que acontece quando aumentamos `q`?
3. O comportamento observado combina com O(n), O(log n) e O(1) médio?
4. Por que não devemos anunciar um “número universal de consultas” em que uma estratégia passa a vencer outra?


## Atividade 4 — mudamos a pergunta: busca por intervalo

Agora queremos:

> **todos os códigos lexicograficamente entre `"22000"` e `"22999"`**

Essa não é a mesma operação da Atividade 2.

Antes de implementar, pense:

- a lista não ordenada sabe onde começa `"22000"`?
- a lista ordenada pode encontrar rapidamente o início da faixa?
- o hashing do `dict` informa quais chaves estão “perto” umas das outras?

### Previsão

> Lista não ordenada →  
> Lista ordenada →  
> `dict` →


In [ ]:
def busca_intervalo_linear(registros, inferior, superior):
    resultado = []
    for codigo, descricao in registros:
        # TODO: acrescente o par se estiver dentro da faixa
        pass
    return resultado


def limite_inferior(registros_ordenados, chave):
    inicio = 0
    fim = len(registros_ordenados)

    while inicio < fim:
        meio = (inicio + fim) // 2

        # TODO:
        # se o código do meio for menor que a chave,
        # descarte a metade esquerda incluindo o meio;
        # caso contrário, preserve o meio como candidato.
        pass

    return inicio


def busca_intervalo_ordenada(registros_ordenados, inferior, superior):
    # TODO:
    # 1. encontre onde a faixa começa;
    # 2. percorra a partir dali;
    # 3. pare assim que codigo > superior.
    return []


inferior, superior = "22000", "22999"

# TODO: implemente também a estratégia com dict varrendo indice.items().
# TODO: verifique se as três estratégias recuperam os mesmos pares.
# Compare como conjuntos quando a ordem da saída não for a mesma.


### O que esta atividade deve mudar na nossa decisão?

Complete:

> Para chave exata, o hashing ajudava porque…  
> Para intervalo, o hashing não fornece naturalmente…  
> A lista ordenada passou a ser interessante porque…  
> Manter a lista ordenada também pode custar…


## Exploração intermediária — quando a repetição é informação

Até agora usamos uma visão com uma entrada por `StockCode`.

Volte ao dataset transacional e observe `InvoiceNo`.

Uma fatura pode aparecer em várias linhas porque uma venda pode conter vários itens.

Antes de executar o código abaixo, responda:

1. Se eliminarmos `InvoiceNo` repetidos, que informação podemos perder?
2. Uma `list` de linhas preserva essa multiplicidade?
3. O que acontece se usarmos diretamente `vendas[invoice] = produto` várias vezes para a mesma chave?
4. Como poderíamos representar **uma chave associada a vários itens**?


In [ ]:
# Escolha uma fatura com mais de uma linha e observe suas ocorrências.
contagem_faturas = df["InvoiceNo"].value_counts()
invoice_exemplo = contagem_faturas.index[0]

linhas_venda = df.loc[
    df["InvoiceNo"] == invoice_exemplo,
    ["InvoiceNo", "StockCode", "Description", "Quantity"]
]

print("InvoiceNo:", invoice_exemplo)
print("Quantidade de linhas:", len(linhas_venda))
display(linhas_venda.head(10))


In [ ]:
# TODO: construa um dicionário em que:
# chave  = InvoiceNo
# valor  = lista de (StockCode, Quantity)
#
# Dica:
# vendas = {}
# for ...:
#     if invoice not in vendas:
#         vendas[invoice] = []
#     vendas[invoice].append(...)

vendas = {}

# TODO

# Depois inspecione:
# vendas[invoice_exemplo][:10]


### Interprete a representação

Complete:

- A lista de registros preserva __________________________________________.
- Um `set` seria adequado se a pergunta fosse apenas ______________________.
- Um `dict` simples com `InvoiceNo → produto` perde informação porque ______.
- Um `dict` com `InvoiceNo → list` permite ________________________________.

**Ideia importante:** estruturas podem ser compostas. A escolha não precisa ser apenas “lista ou `dict` ou `set`”.


## Atividade 5 — agora o problema é realmente de conjuntos

Nesta atividade:

- `A & B` → interseção: está nos dois;
- `A | B` → união: está em pelo menos um;
- `A - B` → diferença: está em A e não em B;
- `A ^ B` → diferença simétrica: está em exatamente um dos dois.

Não usaremos `set` apenas porque `x in set` é rápido. Um `dict` também pode fazer pertencimento por hashing.

Agora a própria pergunta é conjuntista.

Defina:

- **A** = clientes com compras no Reino Unido;
- **B** = clientes com alguma linha cuja `Quantity >= 10`.

Queremos responder:

- `x in A`
- `A & B`
- `A | B`
- `A - B`
- `A ^ B`

### Antes de programar

Explique com palavras o significado de:

> `A & B` →  
> `A - B` →  
> `A | B` →  
> `A ^ B` →


In [ ]:
validos = df[df["CustomerID"].notna()]

A = set(
    validos.loc[
        validos["Country"] == "United Kingdom",
        "CustomerID"
    ].astype(int)
)

B = set(
    validos.loc[
        validos["Quantity"] >= 10,
        "CustomerID"
    ].astype(int)
)


def intersecao_didatica(A, B):
    resultado = set()
    # TODO: percorra um conjunto e use pertencimento no outro
    return resultado


def diferenca_didatica(A, B):
    resultado = set()
    # TODO: mantenha elementos de A que NÃO pertencem a B
    return resultado


def uniao_didatica(A, B):
    resultado = set(A)
    # TODO: acrescente os elementos de B
    return resultado


# TODO: confira suas implementações contra &, - e |
# TODO: escolha um cliente e explique o trabalho conceitual de "cliente in A".


### Revele o trabalho escondido

Depois da execução, responda:

1. Por que inserir novamente o mesmo cliente não cria uma duplicata?
2. Em `x in A`, qual é o papel do hash?
3. Em `A & B`, por que uma única linha de Python não significa O(1)?
4. Por que `A - B` e `B - A` podem produzir resultados diferentes?
5. Neste problema, por que `set` expressa melhor a intenção do que um `dict` artificial sem valores associados?


## Atividade 6 — o que `list.append()` está escondendo?

Uma `list` do CPython pode ser entendida, de forma simplificada, como um **array dinâmico de referências**.

Há duas ideias diferentes:

- **tamanho lógico**: `len(lista)`;
- **capacidade interna**: espaço reservado para referências.

Antes do experimento, desenhe uma lista com:

- tamanho lógico = 4;
- capacidade conceitual = 7.

Depois responda:

> O que acontece se houver espaço e fizermos `append(x)`?  
> E se não houver mais espaço?  
> Python necessariamente dobra a capacidade?  
> Se removermos metade dos elementos, a memória necessariamente cai pela metade?


In [ ]:
import sys
import matplotlib.pyplot as plt

lista_memoria = []
crescimento = [(len(lista_memoria), sys.getsizeof(lista_memoria))]

# TODO 1:
# acrescente 5.000 inteiros.
# Registre um ponto SOMENTE quando sys.getsizeof(lista_memoria) mudar.

# TODO 2:
# remova os 5.000 elementos com pop().
# Novamente registre somente quando o tamanho em bytes mudar.

# TODO 3:
# mostre as primeiras e últimas mudanças.

# TODO 4:
# faça um gráfico de len(lista) × sys.getsizeof(lista)
# distinguindo crescimento e remoção.

# Atenção:
# sys.getsizeof(lista) NÃO soma recursivamente a memória
# ocupada pelos objetos apontados pela lista.


### Interprete os degraus

Não precisamos descobrir a fórmula interna do CPython.

Procure evidências para estas perguntas:

1. `sys.getsizeof()` muda em todo `append`?
2. O crescimento ocorre em degraus?
3. Isso é compatível com **over-allocation**?
4. Por que um `append` individual pode eventualmente ser caro?
5. Por que ainda descrevemos uma sequência de `append`s como O(1) **amortizado**?
6. Durante as remoções, em quais pontos a estrutura devolveu espaço?


## Atividade 7 — consultar rápido pode exigir manutenção

`bisect.insort` resolve um problema específico: **inserir um novo valor em uma lista já ordenada sem perder a ordenação**.

Antes de medir, separe mentalmente duas etapas:

1. localizar a posição correta → busca binária, O(log n);
2. abrir espaço na `list` e inserir → pode deslocar elementos, O(n).

A pergunta é: **o custo de localizar domina o custo total da inserção?**

Na Atividade 4, a ordenação ajudou a consulta por intervalo.

Agora chegam **200 novos produtos**.

Compare:

- `list.append(item)`;
- `bisect.insort(lista, item)`;
- `dicionario[codigo] = descricao`;
- `conjunto.add(codigo)`.

Antes de medir:

> Qual operação precisa deslocar referências para preservar uma propriedade?  
> Qual mantém apenas códigos e perde a descrição?  
> Em qual caso estamos comparando O(1) amortizado com O(1) médio?


In [ ]:
import bisect

amostra = registros[:min(3000, len(registros))]
novos = [
    (f"Z{i:06d}", f"Produto novo {i}")
    for i in range(200)
]

# TODO: implemente quatro funções.
#
# 1. partir de list(amostra) e usar append
# 2. partir de sorted(amostra) e usar bisect.insort
# 3. partir de dict(amostra) e atribuir chave → descrição
# 4. partir de um set de códigos e usar add
#
# Cada função deve partir de uma cópia nova da mesma amostra.
# Depois meça as quatro com timeit.repeat.

# Importante:
# bisect encontra a posição em O(log n),
# mas inserir no meio de um array dinâmico pode deslocar referências.


### Interprete antes de seguir

A pergunta não é apenas “qual foi mais rápido?”.

Complete:

> `append` preserva… mas não preserva…  
> `insort` preserva… pagando…  
> `dict` favorece…  
> `set` representa apenas…  

Se o sistema precisa simultaneamente de busca exata **e** intervalos frequentes, manter mais de uma representação pode ser uma decisão defensável — mas elas precisam permanecer sincronizadas.


## Atividade 8 — o problema mudou novamente: processamento em massa

Agora não queremos localizar um elemento.

Queremos calcular, para muitas linhas:

`Quantity × UnitPrice`

Compare duas formas:

1. laço explícito em Python;
2. operação sobre arrays NumPy.

Antes de executar:

- qual é o Big-O do laço Python?
- qual é o Big-O da operação NumPy sobre `n` valores?
- se ambos forem O(n), você espera exatamente o mesmo tempo? Por quê?


In [ ]:
precos = df["UnitPrice"].fillna(0).astype(float).tolist()
quantidades = df["Quantity"].fillna(0).astype(float).tolist()


def total_python(p, q):
    resultado = []
    for i in range(len(p)):
        # TODO: acrescente p[i] * q[i]
        pass
    return sum(resultado)


def total_numpy(p, q):
    # TODO: use multiplicação vetorizada e np.sum
    pass


# TODO:
# execute para 1.000, 10.000, 100.000 e até 500.000 observações.
#
# Para cada n:
# - prepare listas p e q;
# - prepare arrays float64;
# - confira se os resultados são numericamente próximos;
# - meça Python e NumPy separadamente.
#
# Não inclua silenciosamente a conversão list → ndarray no tempo
# se a intenção for medir somente a operação. Registre esse custo como preparação.


### Explique, não apenas cronometre

Registre:

1. As duas soluções apresentaram a mesma classe assintótica? __________
2. Os tempos medidos foram iguais? __________
3. O experimento mostra que Big-O determina o tempo em segundos? Explique.
4. Houve custo para converter/preparar os dados para NumPy?
5. Em que situação esse custo de preparação poderia não compensar?

**Não precisamos explicar internamente o `ndarray` nesta semana.** O objetivo é distinguir crescimento assintótico de tempo concreto de execução.


## Atividade 9 — volte à hipótese inicial

Você agora tem evidências sobre:

- busca sequencial;
- busca binária e ordenação;
- hashing;
- consultas por intervalo;
- operações de conjuntos;
- custo de manutenção;
- `append` e realocação;
- processamento vetorizado.

Reescreva suas escolhas.

Para cada operação, informe:

**estrutura escolhida → propriedade útil → custo/limitação → evidência observada**

> Consulta exata →  
> Intervalo →  
> Grupos de clientes →  
> Novas inserções →  
> Processamento em massa →

### Fechamento

Há uma estrutura universalmente melhor?

Se você mantiver duas representações dos mesmos dados, qual novo problema de engenharia aparece?
